In [3]:
import pandas as pd
import sqlite3
from scipy.stats import pearsonr

# 1. Conexión a la base de datos
conexion = sqlite3.connect('northwind.db')

# 2. Consulta SQL con LineTotal
query = '''
SELECT 
    od.OrderID,
    p.ProductName,
    od.ProductID,
    c.CategoryName,
    p.CategoryID,
    od.UnitPrice,
    od.Quantity,
    od.Discount,
    ROUND(od.Quantity * od.UnitPrice * (1.0 - od.Discount), 2) AS LineTotal
FROM "Order Details" od
INNER JOIN Products p ON p.ProductID = od.ProductID
INNER JOIN Categories c on c.CategoryID= p.CategoryID
ORDER BY LineTotal DESC
;
'''

# 3. Cargar datos a DataFrame y exportar CSV principal
df_ordenes_detalle = pd.read_sql_query(query, conexion)
df_ordenes_detalle.to_csv('ordenes_detalle.csv', index=False, encoding='utf-8-sig')

# 4. Definir columnas numéricas para las matrices
cols = ['UnitPrice', 'Quantity', 'Discount', 'LineTotal']

# 5. Crear matrices vacías
matriz_corr = pd.DataFrame(index=cols, columns=cols)
matriz_p_value = pd.DataFrame(index=cols, columns=cols)

# 6. Calcular Correlación y P-value
for fila in cols:
    for columnas in cols:
        r, p_val = pearsonr(df_ordenes_detalle[fila], df_ordenes_detalle[columnas])
        matriz_corr.loc[fila, columnas] = round(r, 3)
        matriz_p_value.loc[fila, columnas] = round(p_val, 3)

# 7. Exportar matrices a CSV
matriz_corr.to_csv('matriz_corr.csv', index=True, encoding='utf-8-sig')
matriz_p_value.to_csv('matriz_p_value.csv', index=True, encoding='utf-8-sig')

